# 99 · Build your cross-document agent

A useful agent experiment starts with a specific retrieval-policy change and
enough structure to see its consequences. Stable source IDs, bounded tools,
fresh state for each question, a fixed final answerer, and separate scoring make
it possible to compare that change with a reference method.

The editable policy below follows an invented evidence chain through the shared
answering adapter. Variants demonstrate unnecessary searches and budget limits,
and the recorded standard IRCoT results provide context for choosing a real
experiment. A separately gated development run connects the tools to ADK with
explicit budgets and usage metering.

## 1. A bounded retrieval policy

Separate the retrieval substrate, policy, final answerer, and scorer. The policy
receives only the question and exposed source text. Gold answers, aliases, and
hop graphs enter scoring after answering.

| Component | Keep or replace | Contract |
| --- | --- | --- |
| Passage IDs + local encoder | Replace for your corpus | Stable original source IDs; pinned encoder/index identity |
| Retrieval/evidence tools | Reuse | Original-question first search, update/search alternation, explicit bounds |
| Policy | Edit here; ADK chooses actions in live mode | Source-supported facts and an explicit next evidence gap |
| Final answerer | Keep fixed for the first comparison | Common source template and five-passage budget |
| Scorer + usage meter | Reuse | Full denominators, provenance, recorded calls/tokens/tool work |

The offline policy below is a deliberately simple, readable rule system over
invented evidence. It is not an LLM, a general entity linker, or a trained
retriever. Its job is to make policy edits and their consequences visible.

## 2. A worked example

### Edit a policy, not the gold answer

The only entity substitution comes from text the tool exposed. The function
looks for the authored-work fact, searches for that entity's birthplace, and
selects both sources if present. Change `next_query` to test another strategy;
never return an intermediate answer from hidden gold annotations.

**Predict first:** what happens if the policy repeats the original question
instead of using the observed author name?

In [1]:
# ruff: noqa: E402 - resolve checkout imports before the first example
import json
import sys
from pathlib import Path


# True runs each model step on MODELS through the proxy and caches the result.
# False replays the reference results from RI #2's release with no model calls.
RUN_EXPENSIVE = False

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "aieng-advanced-rag/aieng").is_dir())
sys.path[:0] = [str(ROOT / "aieng-advanced-rag"), str(ROOT / "implementations/cross_document_synthesis")]
from synthesis_lab.tutorial import read_result, require_compute, show_table


require_compute()

from aieng.advanced_rag.models import choose_models


MODELS = choose_models(show=True)

import re
from types import SimpleNamespace

from synthesis_lab.evaluation import question_scores
from synthesis_lab.ircot import IRCoTAnswerer
from synthesis_lab.tutorial import QUESTION, source_answer, starter_tools, synthetic_corpus


CONFIG = {"rounds": 4, "exposed": 5, "strategy": "follow_entity"}
corpus, vectors, encode = synthetic_corpus()


def teaching_policy(functions, config):
    """Build fresh scripted policy state; ADK replaces this rule policy in live mode."""
    update, retrieve = functions
    settings = dict(config)

    async def arun(payload):
        initial = json.loads(payload)
        response = initial["initial_retrieval"]
        observed = {}
        while "passages" in response:
            observed.update({p["passage_id"]: p for p in response["passages"]})
            author = next((p for p in observed.values() if "wrote Beacon." in p["text"]), None)
            birthplace = next((p for p in observed.values() if "was born in" in p["text"]), None)
            if author and birthplace:
                sources = [author, birthplace]
                update(
                    [{"text": p["text"], "passage_ids": [p["passage_id"]]} for p in sources],
                    [p["passage_id"] for p in sources],
                    "",
                    "sufficient",
                )
                return
            entity = re.search(r"([A-Z]\w*) wrote Beacon\.", author["text"]) if author else None
            if not entity:
                update([], [], "", "insufficient")
                return
            next_query = (
                initial["question"]
                if settings["strategy"] == "repeat_original"
                else f"Where was {entity.group(1)} born?"
            )
            accepted = update(
                [{"text": author["text"], "passage_ids": [author["passage_id"]]}],
                [author["passage_id"]],
                next_query,
                "continue",
            )
            if accepted.get("status") != "continue":
                return
            response = retrieve()

    return SimpleNamespace(arun=arun)


def make_answerer(config):
    """Reuse the shared adapter while changing only the offline retrieval policy."""
    return IRCoTAnswerer(
        corpus.passages,
        vectors,
        encode,
        source_answer,
        lambda functions: teaching_policy(functions, config),
        max_rounds=config["rounds"],
        exposed=config["exposed"],
    )


agent = make_answerer(CONFIG)
answer = await agent.aanswer("starter-first", QUESTION)
trace = agent.traces["starter-first"]
print("Answer:", answer.answer.text or "(abstain)", "| Stop:", trace["stop_reason"])
show_table(
    ["Action", "Query / status", "Exposed / selected IDs"],
    [[s["tool"], s.get("query", s.get("status")), s.get("exposed_ids", s.get("evidence_ids"))] for s in trace["steps"]],
)
assert answer.answer.text == "Oslo"

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)


Answer: Oslo | Stop: sufficient


| Action | Query / status | Exposed / selected IDs |
| --- | --- | --- |
| retrieve | Where was the author of Beacon born? | ['demo::00', 'demo::01', 'demo::02', 'demo::03', 'demo::04'] |
| update_evidence | continue | ['demo::00'] |
| retrieve | Where was Mira born? | ['demo::10', 'demo::00', 'demo::01', 'demo::02', 'demo::03'] |
| update_evidence | sufficient | ['demo::00', 'demo::10'] |

### Score after answering and check state isolation

The answerer receives passages and the original question only. The scorer below
then sees the synthetic reference/support IDs. Reusing an adapter must still
allocate fresh tool/session state for every question; a second call should start
from the original question with its own round budget.

In [2]:
def score_answer(answer, trace):
    """Use scorer-only annotations after the adapter has finished."""
    return question_scores(
        corpus.questions[0],
        {
            "answer": answer.answer.text,
            "abstained": answer.abstained,
            "retrieved_ids": trace["retrieved_ids"],
            "generation_ids": trace["generation_ids"],
            "citation_ids": list(answer.answer.chunk_ids),
        },
    )


print("Synthetic metrics:", score_answer(answer, trace))
second = await agent.aanswer("starter-second", QUESTION)
assert second.answer.text == answer.answer.text
second_trace = agent.traces["starter-second"]
assert second_trace["steps"][0]["query"] == QUESTION
assert len(second_trace["steps"]) == len(trace["steps"])
fresh = starter_tools(rounds=CONFIG["rounds"], exposed=CONFIG["exposed"])
assert not fresh.visible and not fresh.rankings and not fresh.stop_reason
print("Repeated calls start with independent evidence and budgets.")

Synthetic metrics: {'exact_match': 1.0, 'token_f1': 1.0, 'generation_recall': 1.0, 'generation_all_supports': 1.0, 'abstained': 0.0, 'failure': 0.0, 'answered_with_valid_citations': 1.0, 'recall_at_5': 1.0, 'all_supports_at_5': 1.0, 'recall_at_10': 1.0, 'all_supports_at_10': 1.0, 'recall_at_20': 1.0, 'all_supports_at_20': 1.0}
Repeated calls start with independent evidence and budgets.


This rule policy succeeds on a designed two-fact chain. It can fail on ambiguity,
paraphrases, distractors, or evidence that requires more than matching phrases.
An accepted `sufficient` update validates source IDs, not the truth of every
fact. Add counterexamples before transferring the policy to a real corpus.

## 3. Recorded results

Use the measured standard IRCoT arm as a reference for what a real model policy
does. These full-corpus rows are **not results of the rule policy you edited**.
The multi-hop gain and single-hop loss help define a useful experimental question:
can your policy recover missing bridges without adding unnecessary searches?

### Evaluation data and fixed configuration

The table below identifies a saved reference run, not the rule policy you edit here.
The MuSiQue reference searches all 11,656 passages for 1,000 original questions.
The NQ control searches 9,633 passages for its 1,000 questions. Model, encoder,
source revision, prompt identity, and evidence settings travel with the saved
reports. Loading these records makes no model calls and does not recompute the
measurements. New experiments must keep the controls fixed or identify the change.


In [3]:
saved = read_result("evaluation/musique_ircot.json")
spec = saved["spec"]
assert saved["completed"] and saved["coverage_complete"] and len(saved["records"]) == 1000
show_table(
    ["Recorded setting", "Value"],
    [
        ["Answer model", spec["model"]],
        ["Generation endpoint", spec["base_url"]],
        ["Local encoder", spec["embedding"][0]],
        ["Encoder revision", spec["embedding"][1]],
        ["Corpus revision", spec["provenance"]["revision"]],
        ["Final passage budget", spec["bounds"]["generation_passages"]],
    ],
)
print("Answer-template SHA-256:", spec["answer_prompt_sha256"])
print("Saved configuration SHA-256:", saved["spec_sha256"])

| Recorded setting | Value |
| --- | --- |
| Answer model | Qwen3_6-35B-A3B |
| Generation endpoint | https://proxy.vectorinstitute.ai/v1 |
| Local encoder | BAAI/bge-small-en-v1.5 |
| Encoder revision | 5c38ec7c405ec4b44b94cc5a9bb96e735b38267a |
| Corpus revision | 5ec05b38deecc3318bb432c69865959c56058990 |
| Final passage budget | 5 |

Answer-template SHA-256: 517c503f1380aa40f6d20b1fc0b8f6538bbf367601e79d4bca0d316cea6eea0f
Saved configuration SHA-256: 2ab467122a55d4121b792da8135a5ba7eeaf5e4397c52e9209d9ae1bde63cdc6


In [4]:
summary = read_result("evaluation/summary.json")
show_table(
    ["Corpus", "Standard IRCoT N", "F1", "Failure rate", "Live calls / q", "Cache hits / q"],
    [
        [
            r["corpus"],
            r["scores"]["overall"]["questions"],
            r["scores"]["overall"]["metrics"]["token_f1"],
            r["scores"]["overall"]["metrics"]["failure"],
            r["scores"]["overall"]["usage"]["calls"] / r["scores"]["overall"]["questions"],
            r["scores"]["overall"]["usage"]["cache_hits"] / r["scores"]["overall"]["questions"],
        ]
        for r in summary["reports"]
        if r["method"] == "ircot"
    ],
)

| Corpus | Standard IRCoT N | F1 | Failure rate | Live calls / q | Cache hits / q |
| --- | --- | --- | --- | --- | --- |
| musique | 1000 | 0.4854 | 0.0090 | 6.2620 | 0.0730 |
| nq | 1000 | 0.6036 | 0.0060 | 4.4440 | 0.2200 |

### Write the experiment contract before a large run

| Record | Why it matters |
| --- | --- |
| Corpus revision/digests and selected IDs | Reproduce the substrate and denominator |
| Encoder, answer model/endpoint, prompt identities | Attribute changes to the intended method |
| Policy, bounds, and final evidence settings | Distinguish your variant from the reference |
| Prediction, generation IDs, citations, stop/failure, usage | Replay quality and inspect cost/failures |
| Paired control and bootstrap unit | Compare matched original IDs, including wording pairs |

The full benchmark runner saves each completed question atomically and rejects
configuration-mismatched resumption. Use a new output/configuration identity for
an edited policy. A successful development question is not a completed benchmark.

### A recorded ADK development check

This one-question execution verifies the live wiring with local pinned embeddings,
the Vector proxy, fresh tool state, fixed source answering, and shared metering.
It is a development example, not a quality estimate. The instruction and answer
prompt digests identify what ran; changing the instruction requires a new run.
Inspect the tool sequence and cost before enabling your own variant.


In [5]:
# The run holds MuSiQue question text and the agent's queries, so it ships in
# RI #2's reference release.
from synthesis_lab.reference import starter_development


development = starter_development(models=MODELS, live=RUN_EXPENSIVE).value
print(development["purpose"])
print("Bounds:", development["config"])
record = development["record"]
print("Question ID:", record["question_id"], "| Stop:", record["stop_reason"])
show_table(
    ["Answer", "F1", "Generation all-supports", "Valid cited answer", "Calls", "Tokens", "Tool calls"],
    [
        [
            record["answer"],
            development["scores"]["token_f1"],
            development["scores"]["generation_all_supports"],
            development["scores"]["answered_with_valid_citations"],
            *[record["usage"][key] for key in ("calls", "total_tokens", "tool_calls")],
        ]
    ],
)
show_table(
    ["Tool", "Query / status", "Exposed / selected IDs"],
    [
        [step["tool"], step.get("query", step.get("status", "")), step.get("exposed_ids", step.get("evidence_ids", []))]
        for step in development["steps"]
    ],
)

starter-agent reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              5 calls, 11,207 prompt + 312 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 5 calls, est. $0.00 (set RUN_EXPENSIVE = True to rebuild)
One-question ADK development check; not a method benchmark
Bounds: {'rounds': 4, 'exposed': 5, 'model_calls': 10}
Question ID: 2hop__13548_13529 | Stop: sufficient


| Answer | F1 | Generation all-supports | Valid cited answer | Calls | Tokens | Tool calls |
| --- | --- | --- | --- | --- | --- | --- |
| June 1982 | 1.0000 | 1.0000 | 1.0000 | 5 | 11519 | 3 |

| Tool | Query / status | Exposed / selected IDs |
| --- | --- | --- |
| retrieve | When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona? | ['musique::00001', 'musique::00005', 'musique::00007', 'musique::00000', 'musique::00016'] |
| update_evidence | continue | ['musique::00001'] |
| retrieve | When was Diego Maradona signed by Barcelona? | ['musique::00002', 'musique::00001', 'musique::00010', 'musique::04056', 'musique::00006'] |
| update_evidence | sufficient | ['musique::00002'] |

## 4. Experiments

### Change one policy decision at a time

Try the baseline, a one-round budget, and a policy that repeats the original
query. Each trial creates fresh tools. **Predict first:** distinguish a retrieval
limit from a repeated-query stop, even if both leave the fixture unable to answer.
The counts below are tool searches in a scripted policy, not paid model calls.

In [6]:
EXPERIMENTS = [
    {"rounds": 4, "exposed": 5, "strategy": "follow_entity"},
    {"rounds": 1, "exposed": 5, "strategy": "follow_entity"},
    {"rounds": 4, "exposed": 5, "strategy": "repeat_original"},
]
rows = []
for index, config in enumerate(EXPERIMENTS):
    variant = make_answerer(config)
    result = await variant.aanswer(f"variant-{index}", QUESTION)
    run_trace = variant.traces[f"variant-{index}"]
    score = score_answer(result, run_trace)
    rows.append(
        [
            config["strategy"],
            config["rounds"],
            run_trace["stop_reason"],
            sum(s["tool"] == "retrieve" for s in run_trace["steps"]),
            result.answer.text or "(abstain)",
            score["token_f1"],
        ]
    )
show_table(["Strategy", "Round limit", "Stop", "Searches used", "Answer", "Synthetic F1"], rows)

| Strategy | Round limit | Stop | Searches used | Answer | Synthetic F1 |
| --- | --- | --- | --- | --- | --- |
| follow_entity | 4 | sufficient | 2 | Oslo | 1.0000 |
| follow_entity | 1 | retrieval_limit | 1 | Not stated in the provided sources. | 0.0000 |
| repeat_original | 4 | repeated_query | 1 | Not stated in the provided sources. | 0.0000 |

### Add a counterexample

For an extension, add a second “Mira” with a different birthplace, a paraphrase
that avoids “wrote”, or an irrelevant birthplace passage. Keep the expected
answer with the scorer. Does your policy's sufficiency test still select the
right sources? Document what extra disambiguating evidence would be needed.
Keep development examples separate from held-out evaluation questions.

## 5. Optional live execution

The recorded check in section 3 and the cell below both call
`starter_development`, one step behind `load_or_run`. With the default
instruction, question, and bounds and `RUN_EXPENSIVE = False`, they replay the
reference built with `Qwen3_6-35B-A3B` from RI #2's release. With
`RUN_EXPENSIVE = True` the run uses `MODELS` and is cached. Any edit below has no
reference, so it needs `RUN_EXPENSIVE = True`.

This one-question **development run** replaces the rule policy with the actual
shared ADK loop. `LIVE_CONFIG` is applied to the real retrieval/call budgets;
editing the offline `teaching_policy` does not change the LLM's policy. To alter
the live policy, edit and identify the instruction separately from the reference.
Final answering keeps the shared source template and five-passage budget.

The gate needs the bootcamp ADK dependencies, pinned local embeddings, and
Vector proxy credentials. It can download/cache MuSiQue and weights, and incur
cost even on failure. All calls share a per-question meter. The development record is
cached with its settings and prompt identities; it is not fed into the completed
benchmark grid. Use `run_benchmark.py` for matched durable
multi-question evaluation.

Edit `LIVE_INSTRUCTION`, `LIVE_QUESTION_ID`, and `LIVE_CONFIG` below to make
a one-question ADK development experiment. Tool-enforced bounds and final source
validation still apply if you change the instruction. The offline rule policy
and live LLM instruction are separate controls. RI #5's
[starter notebook](../agent_native_retrieval/99_starter_agent.ipynb) provides
another example of inspecting tools before enabling expensive agent runs.


Live chat uses the shared model roles and Vector credential resolver. The
default family is Google; use `ADVANCED_RAG_MODEL_FAMILY=onprem` for Qwen
generation and extraction. Embeddings stay local. Set
`ADVANCED_RAG_SPEND_LIMIT_USD` before execution to limit spending. See the
[provider settings](GUIDE.md#models-credentials-and-spend). Recorded results
retain their original models; fresh comparisons require matched model roles.

In [7]:
LIVE_CONFIG = {"rounds": 4, "exposed": 5, "model_calls": 10}
# Edit the instruction independently of the offline rule policy.
from aieng.advanced_rag.runs import ReferenceMissingError
from synthesis_lab.ircot import IRCOT_INSTRUCTION


LIVE_INSTRUCTION = IRCOT_INSTRUCTION
LIVE_QUESTION_ID = None  # Use an existing pinned ID, or None for the reference question.
try:
    mine = starter_development(
        instruction=LIVE_INSTRUCTION,
        config=LIVE_CONFIG,
        question_id=LIVE_QUESTION_ID,
        models=MODELS,
        live=RUN_EXPENSIVE,
    )
except ReferenceMissingError:
    print("No reference for this instruction, question, and bounds. Set RUN_EXPENSIVE = True to run it.")
else:
    print("Development metrics:", mine.value["scores"], "| Usage:", mine.value["record"]["usage"])

starter-agent reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              5 calls, 11,207 prompt + 312 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 5 calls, est. $0.00 (set RUN_EXPENSIVE = True to rebuild)
Development metrics: {'exact_match': 1.0, 'token_f1': 1.0, 'generation_recall': 1.0, 'generation_all_supports': 1.0, 'abstained': 0.0, 'failure': 0.0, 'answered_with_valid_citations': 1.0, 'recall_at_5': 1.0, 'all_supports_at_5': 1.0, 'recall_at_10': 1.0, 'all_supports_at_10': 1.0, 'recall_at_20': 1.0, 'all_supports_at_20': 1.0} | Usage: {'question_id': '2hop__13548_13529', 'calls': 5, 'prompt_tokens': 11207, 'completion_tokens': 312, 'total_tokens': 11519, 'tool_calls': 3, 'cache_hits': 0, 'empty_responses': 0, 'calls_missing_usage': 0, 'service_seconds': 13.792, 'elapsed_seconds': 13.891}


## 6. Interpretation

You now have an editable policy, a bounded answering adapter, synthetic failure
cases, and an evaluation contract. Next, move one hypothesis into a versioned
method, run offline checks, run matched smoke questions, then evaluate complete
held-out cohorts with pinned annotations and paired comparisons.

Before sharing a result:

1. Explain what the policy changed and what remained fixed.
2. Show a success and a counterexample, with original source IDs.
3. Include abstentions, execution failures, stop reasons, and recorded usage.
4. Compare to dense and restored decomposition on identical IDs.
5. Keep graph/article/rewrite and shared-pattern dependencies explicit until supplied.

Use the [decision grid](05_decision_grid.ipynb) to decide where your change helps,
rather than presenting one successful answer as evidence of a general gain.

Keep gold answers and hop graphs with the scorer. Local pinned embeddings and
the Vector proxy are used for live RI #2 experiments. Source IDs establish
provenance; they do not by themselves establish factual entailment.

See the [participant guide](GUIDE.md), [corpus/provenance record](CORPUS.md),
[experiment contracts](docs/EXPERIMENTS.md), and [full matched comparison](docs/COMPARISONS.md).